# Case Study §8 — Unsloth vs HuggingFace, head-to-head

Runnable twin of [`08_unsloth_vs_hf.py`](08_unsloth_vs_hf.py). Same small SFT workload, two backends,
measured: **wall time, peak VRAM, trainable params**.

**The workflow difference is the headline:** Unsloth pins `trl`/`xformers`, so it lives in `.venv`; the
HF/PEFT path lives in `.venv-rl`. One process can't import both — so the script measures the backend in
the current interpreter, then **shells out to the other venv** and prints the comparison.

> **Run this notebook with the `.venv-rl` kernel** (it shells out to `.venv` for the Unsloth half).
> Honesty note: at 135M / a few steps, Unsloth's setup overhead can make it *slower* while still using
> less VRAM; its speed advantage shows up on larger models and longer runs. Measure, don't assume.

In [ ]:
MODE = "trial"     # "trial" or "full"
FORCE = False
import importlib.util, pathlib, sys, json
HERE = pathlib.Path.cwd()
root = HERE if (HERE / "08_unsloth_vs_hf.py").exists() else HERE / "case_study"
sys.path.insert(0, str(root))
import config; config.set_mode(MODE)
spec = importlib.util.spec_from_file_location("s8", root / "08_unsloth_vs_hf.py")
s8 = importlib.util.module_from_spec(spec); spec.loader.exec_module(s8)
print(f"mode={config.RUN_MODE} | current backend = {s8._current_backend()}")

In [ ]:
s8.run(force=FORCE)

## Verify

In [ ]:
import json
hf = json.loads(s8._bench_path('hf').read_text())
print('HF      :', f"{hf['seconds']:.2f}s, {hf['peak_vram_gb']:.2f} GB")
up = s8._bench_path('unsloth')
if up.exists():
    us = json.loads(up.read_text())
    print('Unsloth :', f"{us['seconds']:.2f}s, {us['peak_vram_gb']:.2f} GB")
    assert us['peak_vram_gb'] > 0 and hf['peak_vram_gb'] > 0
assert hf['seconds'] > 0
print('\u2713 §8 verified: same workload benchmarked; interpret speed at scale, not at 135M/few-steps.')
print('Next: Part B \u2014 GGUF \u2192 Ollama \u2192 edge benchmark \u2192 lm-eval-harness/agent.')